Домашнее задание.

ДЗ №6 «Анализ новостей»

Цель:

Студенты учатся использовать сложные предобученные модели для улучшения модели.
Цель - Собрать модель с LLM для оценки тональности новостей.


Описание/Пошаговая инструкция выполнения домашнего задания:

- Добавить в модель анализ тональности новостей и их влияние на позиции из портфеля;
- Выполнить оценку качества модели.

Загружаем используемые библиотеки

In [1]:
#Копия домашнего задания - изменяю PatchTSMixed
#дополняю по результатам проверки
!pip3 install finnhub-python
!pip3 install -U sentence-transformers torch
#!pip3 install backtesting
#!pip3 install bokeh==3.8.1
!pip3 install intel_extension_for_pytorch
!pip3 install pandas-ta
!pip3 install torchmetrics
!pip3 install TA-Lib
!pip3 install ffn
!pip3 install shap

In [2]:
from sentence_transformers import SentenceTransformer
from transformers import PatchTSMixerConfig, PatchTSMixerForPrediction, AutoModel, AutoTokenizer, T5Tokenizer, T5ForConditionalGeneration
from torchmetrics.classification import BinaryAUROC

import finnhub
import pandas as pd
from datetime import datetime, timedelta
import time

import numpy as np
import re
import os

import yfinance as yf
from datetime import datetime

import torch.nn as nn
import torch.optim as optim
import torch
from torch.utils.data import TensorDataset, DataLoader
import shap
import ffn



In [3]:

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import matplotlib.pyplot as plt
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import joblib
from scipy.fft import fft
from IPython.display import display
import pywt
from scipy.stats import skew, kurtosis
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Finbert для CPU (очень медленно)

In [ ]:
# Finbert для CPU
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd

device = "cpu"
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_finbert = AutoModel.from_pretrained(model_name).to(device)

# --- ОПТИМИЗАЦИЯ ДЛЯ CPU В PYTORCH 2.9 ---

# 1. Динамическое квантование (самый мощный прирост на CPU)
# Оно не требует IPEX и отлично работает в PyTorch 2.9
model_finbert = torch.ao.quantization.quantize_dynamic(
    model_finbert, {torch.nn.Linear}, dtype=torch.qint8
)

# 2. Компиляция под CPU (Inductor backend)
# В PyTorch 2.9 это заменяет многие функции IPEX
model_finbert = torch.compile(model_finbert, backend="inductor")

model_finbert.eval()

# 1. Динамическое квантование (уменьшает веса с float32 до int8)
# Это дает 2-3 кратное ускорение на CPU без значительной потери точности
#model_finbert = torch.quantization.quantize_dynamic(
#    model_finbert, {torch.nn.Linear}, dtype=torch.qint8
#)

# 2. Оптимизация IPEX
# model_finbert = ipex.optimize(model_finbert, dtype=torch.float32)
# Примечание: Квантованную модель ipex.optimize обрабатывает автоматически

def finbert_embedding_cpu(sentences, batch_size=16): # Уменьшаем батч для CPU
    if isinstance(sentences, pd.Series):
        sentences = sentences.tolist()

    all_vectors = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            # На CPU не используем .half(), так как мы применили квантование
            outputs = model_finbert(**inputs)

            # Mean Pooling
            attention_mask = inputs['attention_mask']
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            batch_embeddings = sum_embeddings / sum_mask
            batch_embeddings = torch.nn.functional.normalize(batch_embeddings, p=2, dim=1)

            all_vectors.extend(batch_embeddings.numpy().tolist())

    return pd.Series(all_vectors)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

/tmp/ipython-input-1842471941.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_finbert = torch.ao.quantization.quantize_dynamic(


Finbert для CUDA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "ProsusAI/finbert"

# Загрузка токенизатора и модели
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_finbert = AutoModel.from_pretrained(model_name).to(device)

# ОПТИМИЗАЦИЯ 2026: Перевод в полуточность и компиляция (Triton Backend)
model_finbert = model_finbert.half() # Экономия 50% VRAM
model_finbert = torch.compile(model_finbert) # Ускорение инференса через Triton
model_finbert.eval()

def finbert_embedding_text(sentences, batch_size=32):
    if isinstance(sentences, pd.Series):
        sentences = sentences.tolist()

    all_vectors = []

    # Обработка батчами
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]

        # Токенизация с динамическим паддингом (быстрее, чем фиксированный)
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128, # Для новостей 128 обычно достаточно
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            # Инференс
            outputs = model_finbert(**inputs)

            # MEAN POOLING: Усреднение скрытых состояний для получения одного вектора
            # Это стандарт для получения качественного эмбеддинга из BERT
            attention_mask = inputs['attention_mask']
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            batch_embeddings = sum_embeddings / sum_mask

            # Нормализация (как в BGE)
            batch_embeddings = torch.nn.functional.normalize(batch_embeddings, p=2, dim=1)

            # Перенос на CPU и очистка памяти
            all_vectors.extend(batch_embeddings.cpu().to(torch.float32).numpy().tolist())

            # Явная очистка текущих тензоров батча
            del outputs, inputs, batch_embeddings

    print(f"Обработано векторов: {len(all_vectors)}")
    print(f"Размерность вектора FinBERT: {len(all_vectors[0])}") # Будет 768

    return pd.Series(all_vectors)

DistilBART - сочетает мощь CNN-обучения и скорость base-моделей - не использовал

In [ ]:
import torch
import pandas as pd
from transformers import BartTokenizer, BartForConditionalGeneration
import numpy as np

# --- 1. Настройка устройства ---
device = "cpu"

# Используем DistilBART - она сочетает мощь CNN-обучения и скорость base-моделей
model_name_BART = 'sshleifer/distilbart-cnn-12-6'
tokenizer = BartTokenizer.from_pretrained(model_name_BART)
model = BartForConditionalGeneration.from_pretrained(model_name_BART)

# --- 2. Динамическое квантование для CPU ---
# Ускоряет инференс в 2-3 раза за счет перевода весов в int8
model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

model.eval()

def bart_summarize_text_cpu(text):
    if isinstance(text, pd.Series):
        texts = text.fillna("").astype(str).to_list()
    else:
        texts = [str(text)]

    # Токенизация
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        max_length=512, # 512 обычно достаточно для финансовых новостей
        truncation=True
    ).to(device)

    with torch.no_grad():
        # Параметры генерации для DistilBART
        summary_ids = model.generate(
            **inputs,
            max_new_tokens=70,
            min_length=15,
            num_beams=2,         # Оптимально для баланса скорость/качество на CPU
            length_penalty=2.0,  # Возвращаем стандартный штраф для суммаризации
            no_repeat_ngram_size=3,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    summary = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
    return pd.DataFrame(summary, columns=['summary'])

# --- 3. Тест ---
financial_report = """
NVIDIA (NVDA) reports record-breaking revenue for 2024, driven by massive demand for AI chips.
The company's stock surged 15% in pre-market trading as analysts raise price targets.
CEO Jensen Huang stated that the generative AI era is just beginning.
"""
summary_df = bart_summarize_text_cpu(financial_report)
print("Результат суммаризации:")
print(summary_df['summary'].iloc[0])


ValueError: Name tf.RaggedTensorSpec has already been registered for class tensorflow.python.ops.ragged.ragged_tensor.RaggedTensorSpec.

bart-large-cnn - использовалась для суммаризации записей (2022 - 2025 kaggle - 38тыс новостей, 2025 - 2026 finnhub - 11тыс новостей)

In [ ]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration # Изменяем импорт
import pandas as pd

device_map = "cuda" if torch.cuda.is_available() else "cpu"

# --- ИЗМЕНЕНИЕ 1: Загрузка модели BART вместо T5 ---
model_name_BART = 'facebook/bart-large-cnn'
tokenizer = BartTokenizer.from_pretrained(model_name_BART)
model_T5 = BartForConditionalGeneration.from_pretrained(model_name_BART) # Используем ту же переменную model_T5 для минимизации изменений ниже

# --- ОПТИМИЗАЦИЯ ДЛЯ COLAB/GPU ---
model_T5 = model_T5.to(device_map)
model_T5 = model_T5.half() # Переводим в FP16 для экономии VRAM
model_T5 = torch.compile(model_T5) # Опционально: ускорение через Triton compile
model_T5.eval()

# Пример текста финансового отчета (оставляем как есть)
financial_report = """
Tesla has posted strong earnings in the first quarter of 2024, reporting a revenue of $25 billion,
up 15% compared to the same period last year. The company announced that its Model Y sales grew by
20%, contributing to the overall revenue surge. Despite the positive results, Tesla faces challenges
in scaling up its production of batteries, which could affect future growth prospects. Tesla’s stock price
rose by 10% in the aftermath of the earnings report, reflecting investor optimism. Analysts are closely
monitoring the company’s expansion in Europe and its plans to build new factories in Asia.
"""

# Функция для суммирования
def bart_summarize_text(text):
    if isinstance(text, pd.Series):
        texts = text.to_list()
    else:
        # Если передан одиночный текст (как в примере financial_report)
        texts = [text]

    # --- ИЗМЕНЕНИЕ 2: Для BART не требуется префикс "summarize: " ---
    # texts = ["summarize: " + str(t) for t in texts] # Комментируем эту строку

    inputs = tokenizer(texts, return_tensors="pt", padding=True, max_length=1024, truncation=True).to(device_map)

    # Переводим входные данные в float16 (уже сделано при загрузке модели, но здесь перестраховка)
    if model_T5.dtype == torch.float16:
        inputs = {k: v.half() if v.dtype == torch.float32 else v for k, v in inputs.items()}

    # Генерация краткого текста
    with torch.no_grad():
      summary_ids = model_T5.generate(
              **inputs,
              max_new_tokens=150,
              min_length=10,
              length_penalty=2.0,
              num_beams=4, # Для BART-Large 4 луча дают лучшее качество, чем 2
              early_stopping=True,
              pad_token_id=tokenizer.pad_token_id,
              eos_token_id=tokenizer.eos_token_id,
              decoder_start_token_id=model_T5.config.decoder_start_token_id,
              forced_eos_token_id=tokenizer.eos_token_id
          )

    # Декодирование результата
    summary = tokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

    # Очистка тензоров из памяти GPU
    del inputs
    del summary_ids
    torch.cuda.empty_cache() # Дополнительная очистка VRAM

    return pd.DataFrame(summary, columns=['summary'])

# Пример использования с одиночным текстом
summary_df = bart_summarize_text(financial_report)
print(summary_df.iloc[0]['summary'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Tesla has posted strong earnings in the first quarter of 2024, reporting a revenue of $25 billion. The company announced that its Model Y sales grew by 20%. Despite the positive results, Tesla faces challengesin scaling up its production of batteries.


Обработка новостей с kaggle

In [ ]:
from google.colab import drive, files

import gc
import os
import csv
# 1. Set this BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

file_path_kaggle = '/content/drive/MyDrive/Finance/NVDA_news_2022_2025_sub.csv'
file_path_new_kaggle = '/content/drive/MyDrive/Finance/NVDA_kaggle.csv'
try:
    df_kaggle = pd.read_csv(file_path_kaggle,
                            parse_dates=['time_published'], # Автоматически делает datetime64[ns]
                            on_bad_lines='skip',
                            encoding='utf-8-sig')
    print(df_kaggle.head())
    print(f"MIN: {df_kaggle['time_published'].min()}")
    print(f"MAX: {df_kaggle['time_published'].max()}")
    print(f"Count: {len(df_kaggle)}")
    print(f"Файл {file_path_kaggle} успешно загружен!")
except Exception as e:
    print(f"Ошибка при загрузке {file_path_kaggle} : {e}")

df_kaggle['time_published'] = pd.to_datetime(df_kaggle['time_published'], unit='s', errors='coerce')
df_kaggle = df_kaggle.sort_values(by='time_published', ascending=False)
print(f"Post MIN: {df_kaggle['time_published'].min()}")
print(f"Post MAX: {df_kaggle['time_published'].max()}")
print(f"Count: {len(df_kaggle)}")
df_kaggle = df_kaggle[['time_published', 'Summary']]
df_kaggle = df_kaggle.rename(columns={'time_published': 'Date'})

df_kaggle['Summary'] = df_kaggle['Summary'].str[:512].apply(lambda x: f'"""{x}"""')
df_kaggle['Summary'] = df_kaggle['Summary'].str.replace(r'\u200b', '', regex=True).str.replace(r'[\x00-\x1F\x7F-\x9F]', '', regex=True).str.strip()
df_kaggle = df_kaggle.dropna()

print(df_kaggle.head())
print(f"Pred save MIN: {df_kaggle['Date'].min()}")
print(f"Pred save MAX: {df_kaggle['Date'].max()}")
print(f"Count: {len(df_kaggle)}")

df_kaggle.to_csv(file_path_new_kaggle, index=False, encoding='utf-8-sig')
print(f"Данные news успешно сохранены в: {file_path_new_kaggle}")
#
# MIN: 2022-03-03 21:03:36
# MAX: 2025-04-05 19:31:48
# Count: 38820
#



Mounted at /content/drive
                                             Summary      time_published
0  [Sponsored Article]\r\rThe Covid-19 pandemic h... 2022-03-03 21:03:36
1  A lot of money continues to flow into hedge fu... 2022-03-04 17:22:00
2  There are plenty of ways to invest in blockcha... 2022-03-04 19:36:00
3  The chip giant was recently the victim of a hack. 2022-03-06 00:00:00
4  Multinational companies grapple with a fractur... 2022-03-07 11:13:11
MIN: 2022-03-03 21:03:36
MAX: 2025-04-05 19:31:48
Count: 38820
Файл /content/drive/MyDrive/Finance/NVDA_news_2022_2025_sub.csv успешно загружен!
Post MIN: 2022-03-03 21:03:36
Post MAX: 2025-04-05 19:31:48
Count: 38820
                     Date                                            Summary
38819 2025-04-05 19:31:48  """Elon Musk's fortune takes a $12 billion hit...
38818 2025-04-05 14:32:00  """The tech sell-off of 2025 has created sever...
38817 2025-04-05 14:04:00  """Investing in the best growth stocks in the ...
38816 2025

Суммаризация новостей с kaggle и finnhub c помощью bart-large-cnn

In [ ]:

from google.colab import drive, files

import gc
import os
import csv
# 1. Set this BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from pathlib import Path

drive.mount('/content/drive', force_remount=True)
#files.download('temp_data.csv')
file_path = '/content/drive/MyDrive/Finance/my_news_data.csv'

file_path_sentences = '/content/drive/MyDrive/Finance/my_sentences.csv'
file_path_new_kaggle = '/content/drive/MyDrive/Finance/NVDA_kaggle.csv'
file_path_summ_news = '/content/drive/MyDrive/Finance/NVDA_summ_news_20220303_20260111.csv'

# Инициализация клиента Finnhub (замените на ваш ключ)
# finnhub_client = finnhub.Client(api_key="ВАШ_API_КЛЮЧ")
finnhub_client = finnhub.Client(api_key="xxx")


def fetch_news_weekly_chunks(ticker, days_back=365, start_date_full=datetime.strptime('2025-01-17', '%Y-%m-%d'), end_date_full=datetime.strptime('2026-01-11','%Y-%m-%d')):
    all_news = []

    # Определяем общий диапазон
    #end_date_full = datetime.now()
    #start_date_full = end_date_full - timedelta(days=days_back)

    current_start = start_date_full

    print(f"Начало сбора данных за {days_back} дней, разбивая на недельные чанки...")

    # Цикл с шагом в 7 дней
    while current_start < end_date_full:
        current_end = current_start + timedelta(days=7)

        # Если конечная дата выходит за общий диапазон, ограничиваем ее
        if current_end > end_date_full:
            current_end = end_date_full

        # Форматируем даты для API Finnhub (YYYY-MM-DD)
        from_str = current_start.strftime('%Y-%m-%d')
        to_str = current_end.strftime('%Y-%m-%d')

        #print(f"  Запрос интервала: {from_str} to {to_str}")

        try:
            # Запрос к API
            news_batch = finnhub_client.company_news(ticker, _from=from_str, to=to_str)
            all_news.extend(news_batch)

            #print(f"    Получено новостей: {len(news_batch)}")

            # Пауза, чтобы не превысить лимит запросов в минуту (60/мин на free-tier)
            time.sleep(1)

        except Exception as e:
            print(f"    Ошибка при запросе интервала {from_str} - {to_str}: {e}")
            break

        # Сдвигаем начальную дату для следующей итерации
        current_start = current_end

    # Обработка результатов
    if not all_news:
        return pd.DataFrame()

    df = pd.DataFrame(all_news)
    df.drop_duplicates(subset='id', inplace=True) # Удаляем дубликаты, если есть пересечения
    df['datetime'] = pd.to_datetime(df['datetime'], unit='s', errors='coerce')
    df = df.sort_values(by='datetime', ascending=False)

    #return df[['datetime', 'headline', 'source', 'url', 'summary']]
    return df[['datetime', 'summary']]


print("="*70)
# Подготовка данных из Pandas
try:
    df_full_year = pd.read_csv(file_path)
    print("Файл успешно загружен!")
except Exception as e:
    print(f"Ошибка при загрузке news: {e}")

if df_full_year.empty:
     # Загрузка новостей NVDA за последний год (365 дней)
     # df_full_year = fetch_news_weekly_chunks('NVDA', days_back=365)
     if not df_full_year.empty:
        print(f"\nИтого уникальных новостей за год: {len(df_full_year)}")
    #   pd.set_option('display.max_colwidth', None)
        df_full_year['summary'] = df_full_year['summary'].str[:512].apply(lambda x: f'"""{x}"""')
        df_full_year['summary'] = df_full_year['summary'].str.replace(r'\u200b', '', regex=True).str.replace(r'[\x00-\x1F\x7F-\x9F]', '', regex=True).str.strip()
        # df_full_year.to_csv(file_path, index=False, encoding='utf-8-sig')
        print(f"Данные news успешно сохранены в: {file_path}")
    ##df_full_year_news = df_full_year['summary'].apply(lambda x: summarize_text(str(x)) if len(str(x).strip()) > 0 else "") - долго, не батч
    ##texts = ["summarize: " + str(t) for t in df_full_year['summary'].to_list()]
    ###df_full_year['summary'] = summarize_text(df_full_year['summary'])

df_kaggle_news = pd.read_csv(file_path_new_kaggle,
                         on_bad_lines='skip',
                         quoting=csv.QUOTE_MINIMAL,
                         encoding='utf-8-sig')
df_full_year = df_full_year.rename(columns={'datetime': 'Date'})
df_full_year = df_full_year.rename(columns={'summary': 'Summary'})
df_full_year = pd.concat([df_full_year, df_kaggle_news])
df_full_year['Date'] = pd.to_datetime(df_full_year['Date'], errors='coerce')
df_full_year = df_full_year.sort_values(by='Date', ascending=False)
df_full_year['Summary'] = df_full_year['Summary'].str[:512].apply(lambda x: f'"""{x}"""')
df_full_year['Summary'] = df_full_year['Summary'].str.replace(r'\u200b', '', regex=True).str.replace(r'[\x00-\x1F\x7F-\x9F]', '', regex=True).str.strip()
df_full_year['Summary'] = df_full_year['Summary'].str.replace(r"['\"\[\]]", "", regex=True).str.strip()

# 2. Заменяем пустые строки на NaN (чтобы их можно было легко удалить)
df_full_year['Summary'] = df_full_year['Summary'].replace('', np.nan)

# 3. Удаляем строки, где summary теперь пустое (NaN)
df_full_year = df_full_year.dropna(subset=['Summary'])
df_full_year = df_full_year.dropna()

df_full_year = df_full_year.rename(columns={'Summary': 'summary'})
print(df_full_year.head())
print(f"Pred save MIN: {df_full_year['Date'].min()}")
print(f"Pred save MAX: {df_full_year['Date'].max()}")
print(f"Count: {len(df_full_year)}")
df_full_year.to_csv(file_path_summ_news,
                    encoding='utf-8-sig',
                    quoting=csv.QUOTE_MINIMAL # Защита от запятых в тексте
                    )

# Удаляем старый файл, если он существует
output_file = Path(file_path_sentences)
if output_file.exists():
    output_file.unlink()


## Суммаризация новостей
try:
    reader = pd.read_csv(file_path_summ_news, chunksize=100)
    first_batch = True
    with torch.no_grad():
      for batch in reader:
            gc.collect()
            torch.cuda.empty_cache()
            if 'summary' not in batch.columns:
              raise KeyError(f"Колонка 'summary' не найдена. Доступные колонки: {list(batch.columns)}")
            input_texts = batch['summary'].fillna("").astype(str)
            summarized_results = bart_summarize_text(input_texts)
            torch.cuda.empty_cache()
            if isinstance(summarized_results, (pd.DataFrame, pd.Series)):
                batch['summary'] = summarized_results.iloc[:, 0].values if isinstance(summarized_results, pd.DataFrame) else summarized_results.values
            else:
                batch['summary'] = summarized_results
            #print(f"{file_path_sentences}")
            batch.to_csv(file_path_sentences,
                    mode='a',
                    index=False,
                    header=first_batch,
                    encoding='utf-8-sig',
                    quoting=csv.QUOTE_MINIMAL # Защита от запятых в тексте
                    )
            first_batch = False
            del summarized_results, batch
            gc.collect()
            torch.cuda.empty_cache()
    print(f"Суммаризация завершена. Файл с датами сохранен: {output_file}")

except Exception as e:
    print(f"Ошибка при загрузке summary news: {e}")

#Pred save MIN: 2020-06-12 21:14:04
#Pred save MAX: 2026-01-11 23:20:23
#Count: 49857
#Суммаризация завершена. Файл с датами сохранен: /content/drive/MyDrive/Finance/my_sentences.csv




Mounted at /content/drive
Файл успешно загружен!
                 Date                                            summary
0 2026-01-11 23:20:23  X is developing a feature that turns ticker sy...
1 2026-01-11 23:05:00  Nvidia stock has made fortunes for patient inv...
2 2026-01-11 22:50:00  SoundHound continues to post losses, but thats...
3 2026-01-11 22:10:10  PepsiCo, Inc. (NASDAQ:PEP) is included among t...
4 2026-01-11 22:05:00  Archer Aviation recently announced it is colla...
Pred save MIN: 2020-06-12 21:14:04
Pred save MAX: 2026-01-11 23:20:23
Count: 49857
Суммаризация завершена. Файл с датами сохранен: /content/drive/MyDrive/Finance/my_sentences.csv


Предобработка перед созданием embeddings с помощью finbert

In [ ]:
from google.colab import drive
import pandas as pd
import csv

drive.mount('/content/drive', force_remount=True)

file_path_sent_news_raw = '/content/drive/MyDrive/Finance/news_summ2022-2026.csv'
file_path_sentences = '/content/drive/MyDrive/Finance/my_sentences.csv'

df_news_summ = pd.read_csv(file_path_sentences,
                         on_bad_lines='skip',
                         quoting=csv.QUOTE_MINIMAL,
                         encoding='utf-8-sig')

df_news_summ = df_news_summ[['Date','summary']]
df_news_summ['Date'] = pd.to_datetime(df_news_summ['Date'], errors='coerce')
df_news_summ = df_news_summ.sort_values(by='Date', ascending=False)
df_news_summ = df_news_summ.dropna(subset=['Date']) # Удаляем строки с неверными датами

# --- Очистка текста от спецсимволов и кавычек ---
# Используем .str.replace для удаления невидимых, управляющих символов и всех кавычек
df_news_summ['summary'] = df_news_summ['summary'].str.replace(r'\u200b', '', regex=True) \
                                               .str.replace(r'[\x00-\x1F\x7F-\x9F]', '', regex=True) \
                                               .str.replace(r"['\"\[\]]", "", regex=True) \
                                               .str.strip()

# --- Удаление дублирующихся записей ---

# 1. Сначала удаляем строки, где после очистки осталась пустая строка
df_news_summ = df_news_summ[df_news_summ['summary'].str.len() > 0]

# 2. Удаляем точные дубликаты по колонке 'summary'
# Параметр keep='first' оставляет первое вхождение и удаляет остальные
initial_count = len(df_news_summ)
df_news_summ.drop_duplicates(subset=['summary'], keep='first', inplace=True)

final_count = len(df_news_summ)

print(f"Исходное количество строк после базовой очистки: {initial_count}")
print(f"Количество строк после удаления дубликатов: {final_count}")
print(f"Удалено дубликатов: {initial_count - final_count}")

df_news_summ.to_csv(file_path_sent_news_raw,
                    encoding='utf-8-sig',
                    quoting=csv.QUOTE_MINIMAL # Защита от запятых в тексте
                    )

print(f"Данные сохранены в : {file_path_sent_news_raw}")
# Исходное количество строк после базовой очистки: 49857
# Количество строк после удаления дубликатов: 47490
# Удалено дубликатов: 2367

# Готово, теперь df_news_summ готов для вычисления embeddings


Mounted at /content/drive
Исходное количество строк после базовой очистки: 49857
Количество строк после удаления дубликатов: 47490
Удалено дубликатов: 2367
Данные сохранены в : /content/drive/MyDrive/Finance/news_summ2022-2026.csv


Создание embeddings с помощью finbert

In [ ]:
#df_full_year['summary'] = summarize_text(df_full_year['summary'])
#sentences = df_full_year['summary'].fillna("").astype(str).tolist()
#sentences.to_csv(file_path_sentences, index=False, encoding='utf-8-sig')
#print(f"Данные news успешно сохранены в: {file_path_sentences}")
# Принудительный запуск сборщика мусора
#gc.collect()
# Очистка кэша CUDA
#torch.cuda.empty_cache()
from google.colab import drive, files

import gc
import os
import csv
# 1. Set this BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch

drive.mount('/content/drive', force_remount=True)
file_path_sent_news_raw = '/content/drive/MyDrive/Finance/news_summ2022-2026.csv'
file_path_aggregate_embeddings = '/content/drive/MyDrive/Finance/my_aggregate_embeddings.csv'

from pathlib import Path

# Путь к файлу, который нужно создать
output_file = Path(file_path_aggregate_embeddings)

# Удаляем старый файл, если он существует
if output_file.exists():
    output_file.unlink()
    print(f"Старый файл {output_file.name} удален.")
## Embedding новостей
try:
    reader = pd.read_csv(file_path_sent_news_raw, chunksize=500,
                         on_bad_lines='skip',
                         quoting=csv.QUOTE_MINIMAL,
                         encoding='utf-8-sig')
    first_batch = True
    for batch in reader:
            gc.collect()
            torch.cuda.empty_cache()

            sentences = batch['summary'].fillna("no data").astype(str).tolist()

            with torch.no_grad():
                embeddings_data = finbert_embedding_cpu(sentences)

            # 2. Обработка формата возвращаемых данных
            if isinstance(embeddings_data, pd.DataFrame):
                # Если DataFrame имеет 384 колонки, превращаем каждую строку в список (вектор)
                embeddings_list = embeddings_data.values.tolist()
            elif isinstance(embeddings_data, pd.Series):
                embeddings_list = embeddings_data.tolist()
            else:
                embeddings_list = list(embeddings_data)

            # 3. Теперь длина точно совпадает
            batch['embeddings'] = embeddings_list
            batch.to_csv(file_path_aggregate_embeddings,
                    mode='a',
                    index=False,
                    header=first_batch,
                    encoding='utf-8-sig',
                    quoting=csv.QUOTE_MINIMAL # Оборачивает в кавычки только те поля, где есть запятые
                    )
            first_batch = False
            del sentences, embeddings_data, batch
            gc.collect()
            torch.cuda.empty_cache()
    print(f"Данные embeddings успешно сохранены в: {file_path_aggregate_embeddings}")
except Exception as e:
    print(f"Ошибка при загрузке summary news: {e}")

#df_full_year['embeddings'] = embedding_text(sentences)



Mounted at /content/drive


W0116 07:56:53.131000 265 torch/_dynamo/convert_frame.py:1358] [15/8] torch._dynamo hit config.recompile_limit (8)
W0116 07:56:53.131000 265 torch/_dynamo/convert_frame.py:1358] [15/8]    function: 'apply_chunking_to_forward' (/usr/local/lib/python3.12/dist-packages/transformers/pytorch_utils.py:182)
W0116 07:56:53.131000 265 torch/_dynamo/convert_frame.py:1358] [15/8]    last reason: 15/0: Cache line invalidated because L['forward_fn'] got deallocated
W0116 07:56:53.131000 265 torch/_dynamo/convert_frame.py:1358] [15/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0116 07:56:53.131000 265 torch/_dynamo/convert_frame.py:1358] [15/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html
W0116 07:58:15.052000 265 torch/_dynamo/convert_frame.py:1358] [3/8] torch._dynamo hit config.recompile_limit (8)
W0116 07:58:15.052000 265 torch/_dynamo/convert_frame.py:1358] [3/8]    function: 'wrapped_func' (/usr/local/lib/python3.1

Данные embeddings успешно сохранены в: /content/drive/MyDrive/Finance/my_aggregate_embeddings.csv


Проверка embeddings файла

In [ ]:
file_path_aggregate_embeddings = '/content/drive/MyDrive/Finance/my_aggregate_embeddings.csv'

df_emp = pd.read_csv(file_path_aggregate_embeddings,
                         on_bad_lines='skip',
                         quoting=csv.QUOTE_MINIMAL,
                         encoding='utf-8-sig')

print(len(df_emp))
print(df_emp.head())

47490
   Unnamed: 0                 Date  \
0           0  2026-01-11 23:20:23   
1           1  2026-01-11 23:05:00   
2           2  2026-01-11 22:50:00   
3           3  2026-01-11 22:10:10   
4           4  2026-01-11 22:05:00   

                                             summary  \
0  X is developing a feature that turns ticker sy...   
1  Nvidia stock has made fortunes for patient inv...   
2  SoundHound continues to post losses, but thats...   
3  PepsiCo, Inc. (NASDAQ:PEP) is included among t...   
4  Archer Aviation recently announced it is colla...   

                                          embeddings  
0  [0.004110758658498526, -0.0005989112542010844,...  
1  [0.011452561244368553, -0.0038229459896683693,...  
2  [-0.0028355263639241457, 0.03611728549003601, ...  
3  [-0.025234723463654518, 0.012662826105952263, ...  
4  [-0.011249682866036892, 0.03910909965634346, 0...  


Агрегация и конвертация embeddings в векторы

In [ ]:
# Aggregate embeddings
import csv
from google.colab import drive, files
import os
import ast
from pathlib import Path
drive.mount('/content/drive', force_remount=True)
file_path_aggregate_embeddings = '/content/drive/MyDrive/Finance/my_aggregate_embeddings.csv'
file_path_aggregate_embeddings_agg = '/content/drive/MyDrive/Finance/my_aggregate_embeddings_agg.csv'

from pathlib import Path

# Путь к файлу, который нужно создать
output_file = Path(file_path_aggregate_embeddings_agg)

# Удаляем старый файл, если он существует
if output_file.exists():
    output_file.unlink()
    print(f"Старый файл {output_file.name} удален.")

def aggregate_embeddings_by_date(news_data: pd.DataFrame) -> pd.DataFrame:
    # 1. Превращаем строки "[0.1, 0.2...]" обратно в настоящие списки чисел
    # Это критический шаг при чтении из CSV!
    print("Конвертация строк в векторы...")
    news_data['embeddings'] = news_data['embeddings'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # 2. Группируем и считаем среднее
    print("Агрегация по датам...")
    aggregated = news_data.groupby('Date')['embeddings'].apply(lambda x: np.mean(x.tolist(), axis=0))

    aggregated = aggregated.reset_index().rename(columns={'embeddings': 'dailyembedding'})
    return aggregated

# --- Основной процесс ---

# Читаем данные
df_aggregate_embeddings_news = pd.read_csv(
    file_path_aggregate_embeddings,
    on_bad_lines='skip',
    quoting=csv.QUOTE_MINIMAL,
    encoding='utf-8-sig'
)

# Подготовка даты (проверьте, что колонка называется 'datetime')
df_aggregate_embeddings_news['Date'] = pd.to_datetime(df_aggregate_embeddings_news['Date'], utc=True).dt.date

# Агрегация
aggregated_news_data = aggregate_embeddings_by_date(df_aggregate_embeddings_news)


# Разворачиваем список в отдельные колонки (dailyembedding_1, _2...)
print("Разворачивание векторов в колонки...")
# Убеждаемся, что dailyembedding — это массив numpy
embeddings_list = aggregated_news_data['dailyembedding'].tolist()
embeddings_df = pd.DataFrame(embeddings_list)

# Переименовываем
embeddings_df.columns = [f'dailyembedding_{i+1}' for i in range(embeddings_df.shape[1])]

# Соединяем и удаляем промежуточную колонку
aggregated_news_data = pd.concat([aggregated_news_data.drop(columns=['dailyembedding']), embeddings_df], axis=1)

# Финальное сохранение
aggregated_news_data.to_csv(file_path_aggregate_embeddings_agg, index=False, encoding='utf-8-sig')

print("Успешно завершено!")
print(aggregated_news_data.head())

Mounted at /content/drive
Конвертация строк в векторы...
Агрегация по датам...
Разворачивание векторов в колонки...
Успешно завершено!
         Date  dailyembedding_1  dailyembedding_2  dailyembedding_3  \
0  2020-06-12          0.032592          0.005010         -0.003560   
1  2021-04-25          0.034848          0.001084          0.010660   
2  2022-03-03          0.003105          0.038849          0.022148   
3  2022-03-04          0.010861          0.009679          0.010900   
4  2022-03-06         -0.005936         -0.025335         -0.007783   

   dailyembedding_4  dailyembedding_5  dailyembedding_6  dailyembedding_7  \
0          0.042325          0.061536         -0.010158         -0.024016   
1          0.030194          0.051569         -0.046599          0.000932   
2          0.016660          0.028455         -0.029098          0.035579   
3          0.023255          0.056265         -0.023039         -0.023422   
4          0.020165          0.006713         -0.0156

Модель бинарной классификации PatchedTSMixer

In [4]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
import yfinance as yf
import pandas as pd
import pandas_ta as ta # Импортируем pandas-ta
import csv
from google.colab import drive, files
import os
import ast
from pathlib import Path



drive.mount('/content/drive', force_remount=True)
file_path_aggregate_embeddings_agg = '/content/drive/MyDrive/Finance/my_aggregate_embeddings_agg.csv'

ticker = "NVDA"
start_date = "2022-03-03"
end_date = "2026-01-11"

def get_prepared_data(ticker, start, end):
    stock_data = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        interval='1d',
        auto_adjust=True,
        multi_level_index=False,
        ignore_tz=True,
        progress=False
    )
    # ИСПРАВЛЕНИЕ: Если yfinance вернул MultiIndex (например, ('Close', 'NVDA'))
    if isinstance(stock_data.columns, pd.MultiIndex):
        stock_data.columns = stock_data.columns.get_level_values(0)

    stock_data.reset_index(inplace=True)
    stock_data['Returns'] = stock_data['Close'].pct_change()

    # Убеждаемся, что колонка называется 'Date' (иногда yf возвращает 'index')
    if 'Date' not in stock_data.columns:
        stock_data.rename(columns={stock_data.columns[0]: 'Date'}, inplace=True)
    #stock_data = stock_data[['Date', 'Returns', 'Close', 'Volume']]
    # --- ДОБАВЛЕНИЕ ТЕХНИЧЕСКИХ ИНДИКАТОРОВ ---
    stock_data.ta.macd(append=True)
    stock_data.ta.psar(append=True)
    stock_data.ta.vwma(append=True)
    stock_data.ta.rsi(close='Close', length=14, append=True)
    stock_data.ta.macd(fast=6, slow=13, signal=5, suffix='_Dbl1', append=True)
    stock_data.ta.macd(fast=12, slow=26, signal=9, suffix='_Dbl2', append=True)

    stock_data.ta.atr(timeperiod=14, append=True)
    stock_data.ta.cci(timeperiod=14, append=True)
    stock_data.ta.adx(timeperiod=14, append=True)
    stock_data.ta.mfi(timeperiod=14, append=True)
    stock_data.ta.linreg(length=14, angle=True, append=True)
    stock_data.ta.bbands(length=20, std=2, append=True)
    stock_data.ta.sma(length=50, append=True) # Добавляем SMA 50
    stock_data.ta.sma(length=200, append=True) # Добавляем SMA 200
    stock_data.ta.obv(append=True)
    stock_data.ta.adosc(append=True)

    stock_data = stock_data.fillna(0)


    news_data = pd.read_csv(
                file_path_aggregate_embeddings_agg,
                parse_dates=['Date'], # Автоматически делает datetime64[ns]
                on_bad_lines='skip',
                encoding='utf-8-sig'
            )

    # Синхронизация форматов времени (КРИТИЧНО)
    stock_data['Date'] = pd.to_datetime(stock_data['Date']).dt.normalize().dt.tz_localize(None)
    news_data['Date'] = pd.to_datetime(news_data['Date']).dt.normalize().dt.tz_localize(None)

    # Объединение
    # df = pd.merge(stock_data, news_data, on='Date', how='left')
    df = pd.merge(stock_data, news_data, on='Date', how='inner')

    df = df.fillna(0) # Заполнение пропусков в эмбеддингах
    return df

df_combined = get_prepared_data(ticker, start_date, end_date)

# 1. Формирование бинарного таргета (target)
# Предсказываем: вырастет ли цена Close завтра по сравнению с сегодня
df_combined['target'] = (df_combined['Close'].shift(-1) > df_combined['Close']).astype(int)
# Удаляем последнюю строку, так как для нее нет значения завтрашней цены
df_combined = df_combined.dropna().iloc[:-1]

#embedding_cols = [c for c in df_combined.columns if c not in ['Date', 'Returns', 'Close', 'Volume', 'target']]
embedding_cols = [c for c in df_combined.columns if c not in ['Date', 'Returns', 'Close', 'Volume',
       'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'PSARl_0.02_0.2',
       'PSARs_0.02_0.2', 'PSARaf_0.02_0.2', 'PSARr_0.02_0.2', 'VWMA_10',
       'RSI_14', 'MACD_6_13_5__Dbl1', 'MACDh_6_13_5__Dbl1',
       'MACDs_6_13_5__Dbl1', 'MACD_12_26_9__Dbl2', 'MACDh_12_26_9__Dbl2',
       'MACDs_12_26_9__Dbl2', 'ATRr_14', 'CCI_14_0.015', 'ADX_14', 'ADXR_14_2',
       'DMP_14', 'DMN_14', 'MFI_14', 'LINREGa_14', 'BBL_20_2.0_2.0',
       'BBM_20_2.0_2.0', 'BBU_20_2.0_2.0', 'BBB_20_2.0_2.0', 'BBP_20_2.0_2.0',
       'SMA_50', 'SMA_200', 'OBV', 'ADOSC_3_10']]
#stock_cols = ['Date', 'Returns', 'Close', 'Volume', 'target']
stock_cols = ['Date','Returns', 'Close', 'Volume', 'target',
       'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'PSARl_0.02_0.2',
       'PSARs_0.02_0.2', 'PSARaf_0.02_0.2', 'PSARr_0.02_0.2', 'VWMA_10',
       'RSI_14', 'MACD_6_13_5__Dbl1', 'MACDh_6_13_5__Dbl1',
       'MACDs_6_13_5__Dbl1', 'MACD_12_26_9__Dbl2', 'MACDh_12_26_9__Dbl2',
       'MACDs_12_26_9__Dbl2', 'ATRr_14', 'CCI_14_0.015', 'ADX_14', 'ADXR_14_2',
       'DMP_14', 'DMN_14', 'MFI_14', 'LINREGa_14', 'BBL_20_2.0_2.0',
       'BBM_20_2.0_2.0', 'BBU_20_2.0_2.0', 'BBB_20_2.0_2.0', 'BBP_20_2.0_2.0',
       'SMA_50', 'SMA_200', 'OBV', 'ADOSC_3_10']

# 2. PCA и подготовка данных
n_components = 32
pca = PCA(n_components=n_components)
embeddings_reduced = pca.fit_transform(df_combined[embedding_cols])
embeddings_pca_df = pd.DataFrame(
    embeddings_reduced,
    columns=[f'pca_{i}' for i in range(n_components)],
    index=df_combined.index
)
df_pca = pd.concat([df_combined[stock_cols], embeddings_pca_df], axis=1)
df_combined = df_pca.sort_values('Date')

# 3. Разделение на выборки (через TimeSeriesSplit)
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
for train_index, test_index in tscv.split(df_combined):
    train_df = df_combined.iloc[train_index]
    test_df = df_combined.iloc[test_index]

# Определение колонок (target не входит в признаки X)
features_cols = [c for c in df_combined.columns if c not in ['Date', 'target']]
target_col = 'target'

# 4. Масштабирование
scaler = RobustScaler()
train_features_scaled = scaler.fit_transform(train_df[features_cols])
test_features_scaled = scaler.transform(test_df[features_cols])

# 5. Создание последовательностей для TSMixer (Бинарная классификация)
#def create_tsmixer_sequences(data_features, data_target, seq_length):
#    xs, ys = [], []
#    for i in range(len(data_features) - seq_length):
#        x = data_features[i:i+seq_length]
#        # Таргет берется для последнего дня в окне (уже сдвинут на t+1)
#        y = data_target[i+seq_length - 1]
#        xs.append(x)
#        ys.append(y)
#    return torch.tensor(np.array(xs), dtype=torch.float32), torch.tensor(np.array(ys), dtype=torch.float32).view(-1, 1)

# 5. Создание последовательностей. Функция для создания скользящих окон
def create_tsmixer_sequences(X_normalized, y_labels, lookback):
      """
      Создает скользящие окна для задачи классификации временных рядов.

      Параметры:
      - X_normalized: нормализованные признаки (T, num_features)
      - y_labels: метки классов (T,)
      - lookback: размер окна истории

      Возвращает:
      - X_windows: (N, lookback, num_features)
      - y_windows: (N,)
      - indices: индексы в оригинальном массиве
      """
      X_windows, y_windows, indices = [], [], []

      for i in range(lookback, len(y_labels)):
          # Берем окно истории размером lookback
          X_windows.append(X_normalized[i - lookback:i, :])
          # Таргет - это метка в момент времени i
          y_windows.append(y_labels[i])
          indices.append(i)

      X_windows = np.asarray(X_windows, dtype=np.float32)  # (N, lookback, num_features)
      y_windows = np.asarray(y_windows, dtype=np.int64)     # (N,)
      indices = np.asarray(indices, dtype=np.int64)

      return X_windows, y_windows, indices

seq_length = 128
batch_size_loader = 64
X_train, y_train, idx_train = create_tsmixer_sequences(train_features_scaled, train_df[target_col].values, seq_length)
X_test, y_test, idx_test = create_tsmixer_sequences(test_features_scaled, test_df[target_col].values, seq_length)

# 6. DataLoader
#train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size_loader, shuffle=True, drop_last=False)
#test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size_loader, shuffle=False, drop_last=False)

train_dataset = TensorDataset(
      torch.from_numpy(X_train),
      torch.from_numpy(y_train))
test_dataset = TensorDataset(
      torch.from_numpy(X_test),
      torch.from_numpy(y_test))
train_loader = DataLoader(
      train_dataset,
      batch_size=batch_size_loader,
      shuffle=True,
      drop_last=False)
test_loader = DataLoader(
      test_dataset,
      batch_size=batch_size_loader,
      shuffle=False,
      drop_last=False)

# 7. Заглушка структуры PatchedTSMixer для бинарной классификации
# class PatchedTSMixerBinary(nn.Module):
#    def __init__(self, seq_len, n_vars, patch_len=8):
#        super().__init__()
#        # Упрощенная логика Mixers (в реальной библиотеке используйте HuggingFace или специализированную имплементацию)
#        self.linear_backbone = nn.Linear(n_vars * seq_len, 128)
#        self.head = nn.Sequential(
#            nn.ReLU(),
#            nn.Linear(128, 1),
#            nn.Sigmoid() # Для бинарной классификации
#        )
#
#    def forward(self, x):
#        # x shape: [batch, seq_len, n_vars]
#        x = x.view(x.size(0), -1)
#        x = self.linear_backbone(x)
#        return self.head(x)

# Инициализация модели
# model = PatchedTSMixerBinary(seq_len=seq_length, n_vars=len(features_cols))
# criterion = nn.BCELoss() # Бинарная кросс-энтропия
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Входной тензор: {X_train.shape}") # [Записи, 80, 35] (32 PCA + Returns, Close, Volume)
print(f"Целевой тензор: {y_train.shape}") # [Записи, 1]


Mounted at /content/drive
Входной тензор: (662, 128, 67)
Целевой тензор: (662,)


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
# Шаг 1: Подготовка данных для бинарной классификации
# ======================================================

LOOKBACK = seq_length        # сколько дней истории подаем в модель
PATCHLENGTH = LOOKBACK - 1
HORIZON  = 7         # сколько дней прогнозируем вперед
TRAIN_RATIO = 1
EPOCHS = 50
BATCH_SIZE = batch_size_loader
LR = 3e-4
WEIGHT_DECAY = 1e-2
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
tsmixer_output_dir = "tsmixer_full_NVDA.pt"

#=======================================================

#=======================================================
print("=" * 70)
print("РАСПРЕДЕЛЕНИЕ КЛАССОВ")
print("=" * 70)
print(df_combined['target'].value_counts().sort_index())
print(f"\nБаланс классов (доля класса 1): {df_combined['target'].mean():.3f}")
print(f"Всего примеров: {len(df_combined)}")
print("=" * 70)

# 2: Нормализация и создание временных окон
# ================================================

print(f"Размер окон признаков: {X_train.shape}")
print(f"Размер таргетов: {y_train.shape}")

print("=" * 70)
print("РАЗМЕРЫ ВЫБОРОК")
print("=" * 70)
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print("=" * 70)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Проверим размерность одного батча
xb_sample, yb_sample = next(iter(train_loader))
print(f"\nПример батча:")
print(f"  X: {xb_sample.shape}  (batch, lookback, num_features)")
print(f"  y: {yb_sample.shape}  (batch,)")

# Проверим размерность одного батча test
xb_sample_test, yb_sample_test = next(iter(test_loader))
print(f"\nПример батча test:")
print(f"  X: {xb_sample_test.shape}  (batch, lookback, num_features)")
print(f"  y: {yb_sample_test.shape}  (batch,)")

# 6: Создание и инициализация модели
# ========================================

# Конфигурация модели
# ВАЖНО: prediction_length=1, так как мы используем выход как feature vector
config_binary = PatchTSMixerConfig(
    context_length=LOOKBACK,          # Длина входного окна
    prediction_length=1,              # Выход размером 1 (используется как признаки)
    #Обычно для классификации временных рядов prediction_length устанавливается в 0 или игнорируется, если используется PatchTSMixerForClassification
    num_input_channels=len(features_cols), # Количество признаков (12)
    patch_length=PATCHLENGTH,                  # Размер патча
    #patch_stride=8,                   # Шаг между патчами
    patch_stride=PATCHLENGTH,
    #Обычно stride выбирают равным patch_length (без перекрытия) или чуть меньше. С таким маленьким шагом модель будет обучаться очень медленно.
    d_model=96,                       # Размерность скрытого представления
    num_layers=6,                     # Количество слоев
    dropout=0.1,                      # Dropout для регуляризации
)

# =============================================================
# Определение модели классификации

class TSMixerBinaryClassifier(nn.Module):
    """
    Бинарный классификатор на основе PatchTSMixer.

    Архитектура:
    1. Backbone: стандартный PatchTSMixerForPrediction из HuggingFace
    2. Feature extraction: используем prediction_outputs как признаки
    3. Classification head: полносвязные слои с dropout

    Преимущества:
    - Используем проверенную архитектуру из transformers
    - Backbone уже оптимизирован для работы с временными рядами
    - Простота и прозрачность кода
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Backbone: стандартная модель PatchTSMixer для prediction
        # Она обучается предсказывать следующий шаг, и мы используем её выходы как признаки
        self.backbone = PatchTSMixerForPrediction(config)

        # Classification head
        # Входные признаки: (batch, prediction_length=1, num_channels)
        # После flatten: (batch, num_channels)
        self.classifier = nn.Sequential(
            nn.Flatten(),  # (batch, 1, num_channels) -> (batch, num_channels)
            nn.Linear(config.num_input_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)  # 2 класса: падение (0) и рост (1)
        )

    def forward(self, past_values):
        """
        Forward pass.

        Параметры:
        - past_values: (batch, context_length, num_channels)

        Возвращает:
        - logits: (batch, 2) - логиты для двух классов
        """
        # Получаем prediction outputs из backbone
        outputs = self.backbone(past_values=past_values)

        # prediction_outputs имеет размер (batch, prediction_length=1, num_channels)
        # Эти выходы содержат закодированную информацию о временном ряде
        features = outputs.prediction_outputs

        # Пропускаем через classification head
        logits = self.classifier(features)

        return logits

print("Класс TSMixerBinaryClassifier определен")
# =============================================================

# Создаем модель
print("=" * 70)
model_binary = TSMixerBinaryClassifier(config_binary).to(device)

# Считаем количество параметров
total_params = sum(p.numel() for p in model_binary.parameters())
trainable_params = sum(p.numel() for p in model_binary.parameters() if p.requires_grad)

print("=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров:      {total_params:,}")
print(f"Обучаемых параметров:  {trainable_params:,}")
print(f"Устройство:            {device}")
print("=" * 70)

# 7: Настройка обучения
# ===========================

# Optimizer: AdamW с weight decay для регуляризации
print(f"Установлен новый Optimizer: AdamW с weight decay для регуляризации")
optimizer_binary = torch.optim.AdamW(
          model_binary.parameters(),
          lr=LR,
          weight_decay=WEIGHT_DECAY
      )

# -------------------------------
# 1. Считаем количество примеров каждого класса в обучающей выборке
class_counts = df_combined['target'].value_counts().sort_index().values
# class_counts[0] — количество объектов класса 0 (Down)
# class_counts[1] — количество объектов класса 1 (Up)

# 2. Вычисляем веса (обратная пропорция)
# Общая логика: weight = total_samples / (n_classes * class_samples)
total_samples = len(df_combined)
n_classes = 2

weights = total_samples / (n_classes * class_counts)

# 3. Переводим в тензор и отправляем на устройство (CPU/GPU)
weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

# 4. Инициализируем лосс с весами
criterion_binary = nn.CrossEntropyLoss(weight=weights_tensor)

print(f"Веса классов для Loss: 0 (Down): {weights[0]:.4f}, 1 (Up): {weights[1]:.4f}")
# Функция для evaluation
def evaluate_binary_classifier():
      """
      Оценка модели на test set.

      Возвращает:
      - accuracy: точность классификации
      - predictions: предсказанные метки
      - true_labels: истинные метки
      """
      model_binary.eval()
      all_preds = []
      all_labels = []

      with torch.no_grad():
          for xb, yb in test_loader:
              xb = xb.to(device)
              yb = yb.to(device)

              # Forward pass
              logits = model_binary(xb)

              # Предсказания (argmax по логитам)
              preds = torch.argmax(logits, dim=1)

              # Сохраняем для подсчета метрик
              all_preds.extend(preds.cpu().numpy())
              all_labels.extend(yb.cpu().numpy())

      all_preds = np.array(all_preds)
      all_labels = np.array(all_labels)

      accuracy = (all_preds == all_labels).mean()

      return accuracy, all_preds, all_labels

print("✓ Optimizer, loss function и evaluation function настроены")

# 8: Обучение модели
# ========================
# Инициализация для сохранения лучшей модели
best_test_acc = 0.0

print("=" * 70)
print("НАЧАЛО ОБУЧЕНИЯ")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
      # Training phase
      model_binary.train()
      train_loss_sum = 0.0
      train_correct = 0
      train_total = 0

      for xb, yb in train_loader:
          xb = xb.to(device)
          yb = yb.to(device)

          # Forward pass
          logits = model_binary(xb)
          loss = criterion_binary(logits, yb)

          # Backward pass
          optimizer_binary.zero_grad(set_to_none=True)
          loss.backward()

          # Gradient clipping для стабильности
          torch.nn.utils.clip_grad_norm_(model_binary.parameters(), max_norm=1.0)

          # Update weights
          optimizer_binary.step()

          # Статистика
          train_loss_sum += loss.item()
          preds = torch.argmax(logits, dim=1)
          train_correct += (preds == yb).sum().item()
          train_total += yb.size(0)

      # Метрики за эпоху
      train_loss_avg = train_loss_sum / len(train_loader)
      train_acc = train_correct / train_total

      # Evaluation на test set
      test_acc, _, _ = evaluate_binary_classifier()

      # ЛОГИКА СОХРАНЕНИЯ ЛУЧШЕЙ МОДЕЛИ
      if test_acc > best_test_acc:
          best_test_acc = test_acc
          # Сохраняем состояние (state_dict) - самый надежный способ
          #torch.save(model_binary.state_dict(), tsmixer_output_dir)
          checkpoint = {
              'model_state_dict': model_binary.state_dict(),
              'optimizer_state_dict': optimizer_binary.state_dict(),
          }
          torch.save(checkpoint, tsmixer_output_dir)
          save_status = f" | [Model Saved! Best Acc: {best_test_acc:.4f}]"
      else:
          save_status = ""

      # Вывод результатов
      print(f"Epoch {epoch:2d}/{EPOCHS} | "
            f"Train Loss: {train_loss_avg:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Test Acc: {test_acc:.4f}{save_status}")


print("=" * 70)
print(f"ОБУЧЕНИЕ ЗАВЕРШЕНО. Лучшая точность: {best_test_acc:.4f}")
print("=" * 70)

# ЗАГРУЖАЕМ ЛУЧШИЕ ВЕСА ПЕРЕД ФИНАЛЬНОЙ ОЦЕНКОЙ
if os.path.exists(tsmixer_output_dir):
      print("=" * 70)
      print(f"ЗАГРУЖАЕМ ЛУЧШИЕ ВЕСА ПЕРЕД ФИНАЛЬНОЙ ОЦЕНКОЙ")
      checkpoint = torch.load(tsmixer_output_dir, map_location=device)
      model_binary.load_state_dict(checkpoint['model_state_dict'])
      optimizer_binary = torch.optim.AdamW(model_binary.parameters(), lr=LR)
      if 'optimizer_state_dict' in checkpoint:
         optimizer_binary.load_state_dict(checkpoint['optimizer_state_dict'])
      model_binary.to(device)
      print(f"Загружены лучшие веса из {tsmixer_output_dir}")
print("=" * 70)

final_test_acc, y_pred_binary, y_true_binary = evaluate_binary_classifier()

print("\n" + "=" * 70)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ")
print("=" * 70)
print(f"Test Accuracy: {final_test_acc:.4f} ({final_test_acc*100:.2f}%)")

# Baseline - всегда предсказываем мажоритарный класс
baseline_acc = max(y_true_binary.mean(), 1 - y_true_binary.mean())
print(f"Baseline (majority class): {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
print(f"Улучшение над baseline: {(final_test_acc - baseline_acc)*100:.2f}%")

# Classification Report
print("\n" + "-" * 70)
print("CLASSIFICATION REPORT")
print("-" * 70)
print(classification_report(
    y_true_binary,
    y_pred_binary,
    target_names=['Down (0)', 'Up (1)'],
    digits=4
))

# Confusion Matrix
print("-" * 70)
print("CONFUSION MATRIX")
print("-" * 70)
cm = confusion_matrix(y_true_binary, y_pred_binary)
print("\n              Predicted")
print("                0      1")
print(f"Actual 0    {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"       1    {cm[1,0]:5d}  {cm[1,1]:5d}")

# Дополнительные метрики
tn, fp, fn, tp = cm.ravel()
precision_up = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_up = tp / (tp + fn) if (tp + fn) > 0 else 0
precision_down = tn / (tn + fn) if (tn + fn) > 0 else 0
recall_down = tn / (tn + fp) if (tn + fp) > 0 else 0

print("\n" + "-" * 70)
print("ИНТЕРПРЕТАЦИЯ")
print("-" * 70)
print(f"True Positives (правильно предсказан рост):     {tp}")
print(f"True Negatives (правильно предсказано падение): {tn}")
print(f"False Positives (ложный сигнал на рост):        {fp}")
print(f"False Negatives (пропущенный рост):             {fn}")

# 11: Бэктест торговых стратегий
# ====================================

# Рассчитаем реальные returns
actual_returns_binary = np.zeros(len(y_true_binary))

df_combined['y_next_close'] = df_combined['Close'].shift(-1)
df_combined = df_combined.dropna().iloc[:-1]
for i in range(len(idx_test)):
      idx = idx_test[i]
      if idx < len(df_combined):
          today_price = df_combined["Close"].iloc[idx]
          next_price = df_combined["y_next_close"].iloc[idx]
          actual_returns_binary[i] = (next_price - today_price) / today_price

# Стратегия 1: Long only (покупаем только когда модель предсказывает рост)
strategy_long_only = np.where(y_pred_binary == 1, actual_returns_binary, 0.0)

# Стратегия 2: Long/Short (лонг при росте, шорт при падении)
strategy_long_short = np.where(y_pred_binary == 1, actual_returns_binary, -actual_returns_binary)

# Кумулятивные доходности
cum_return_buy_hold = np.cumprod(1 + actual_returns_binary) - 1
cum_return_long_only = np.cumprod(1 + strategy_long_only) - 1
cum_return_long_short = np.cumprod(1 + strategy_long_short) - 1

# Статистика
print("\n" + "=" * 70)
print("BACKTEST ТОРГОВЫХ СТРАТЕГИЙ")
print("=" * 70)
print(f"Buy & Hold:              {cum_return_buy_hold[-1]:7.2%}")
print(f"Strategy (Long only):    {cum_return_long_only[-1]:7.2%}")
print(f"Strategy (Long/Short):   {cum_return_long_short[-1]:7.2%}")
print("=" * 70)

# Дополнительные метрики
sharpe_bh = actual_returns_binary.mean() / (actual_returns_binary.std() + 1e-8) * np.sqrt(252)
sharpe_long = strategy_long_only.mean() / (strategy_long_only.std() + 1e-8) * np.sqrt(252)
sharpe_ls = strategy_long_short.mean() / (strategy_long_short.std() + 1e-8) * np.sqrt(252)

print(f"\nSharpe Ratio (годовой):")
print(f"  Buy & Hold:           {sharpe_bh:.3f}")
print(f"  Strategy (Long only): {sharpe_long:.3f}")
print(f"  Strategy (Long/Short):{sharpe_ls:.3f}")
print("=" * 70)

# Считаем ROC AUC
model_binary.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model_binary(xb)
        # Получаем вероятности класса 1 (Up)
        probs = torch.softmax(logits, dim=1)[:, 1]

        all_probs.append(probs.cpu())
        all_labels.append(yb.cpu())

# Объединяем все батчи в единые тензоры
all_probs = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

# Считаем метрику
metric = BinaryAUROC()
auc_score = metric(all_probs, all_labels)

print(f"Финальный ROC AUC на тесте: {auc_score.item():.4f}")

# =====================================
# Подготавдиваем данные для сравнения с бенчмарком
test_df_bench = test_df[ LOOKBACK: ]
test_df_bench['Signal'] = y_pred_binary

# Добавил вычисление SHAP


РАСПРЕДЕЛЕНИЕ КЛАССОВ
target
0    436
1    508
Name: count, dtype: int64

Баланс классов (доля класса 1): 0.538
Всего примеров: 944
Размер окон признаков: (662, 128, 67)
Размер таргетов: (662,)
РАЗМЕРЫ ВЫБОРОК
Train: X=(662, 128, 67), y=(662,)
Test:  X=(30, 128, 67), y=(30,)
Train batches: 11
Test batches:  1

Пример батча:
  X: torch.Size([64, 128, 67])  (batch, lookback, num_features)
  y: torch.Size([64])  (batch,)

Пример батча test:
  X: torch.Size([30, 128, 67])  (batch, lookback, num_features)
  y: torch.Size([30])  (batch,)
Класс TSMixerBinaryClassifier определен
ИНФОРМАЦИЯ О МОДЕЛИ
Всего параметров:      310,617
Обучаемых параметров:  310,617
Устройство:            cuda
Установлен новый Optimizer: AdamW с weight decay для регуляризации
Веса классов для Loss: 0 (Down): 1.0826, 1 (Up): 0.9291
✓ Optimizer, loss function и evaluation function настроены
НАЧАЛО ОБУЧЕНИЯ
Epoch  1/50 | Train Loss: 0.6955 | Train Acc: 0.5000 | Test Acc: 0.4667 | [Model Saved! Best Acc: 0.4667]
Epoch  2

In [34]:
import pandas as pd
import numpy as np
import yfinance as yf
import ffn
import shap
import torch

#Созданы эмбендинги, использован PCA для уменьшения размеренности (0.74 дисперсии объяснено).
#Эмбендинги использовались в качестве параметров для улучшения точности.
#Кроме того, вижу, что используете наиболее популярную модель GARCH(1,1)
#для оценки волатильности, которая реагирует на шоки.

#Для оценки влияния признаков на модель можно пробовать использовать SHAP.
#Можно будет визуально оценить, насколько влияют на предсказания полученные эмбендинги.

# Определение устройства
device = next(model_binary.parameters()).device

# --- 1. Подготовка сигналов ---
LOOKBACK = seq_length
test_df_bench = test_df.iloc[LOOKBACK:].copy()

if isinstance(y_pred_binary, torch.Tensor):
    test_df_bench['Signal'] = y_pred_binary.detach().cpu().numpy().flatten()
else:
    test_df_bench['Signal'] = y_pred_binary.flatten()

# --- 2. Исправленный SHAP (исправление несовпадения длин) ---
print("--- 🧠 Вычисление SHAP Values ---")
try:
    # 1. Подготовка тензоров на правильном устройстве
    X_train_tensor = torch.as_tensor(X_train[:100], dtype=torch.float32).to(device)
    X_test_tensor = torch.as_tensor(X_test[:50], dtype=torch.float32).to(device)

    # 2. Инициализация и расчет
    explainer = shap.GradientExplainer(model_binary, X_train_tensor)
    shap_values = explainer.shap_values(X_test_tensor)

    # 3. Преобразование в numpy и обработка структуры
    # Если на выходе список (для классификации), берем первый элемент
    if isinstance(shap_values, list):
        shap_matrix = np.array(shap_values[0])
    else:
        shap_matrix = np.array(shap_values)

    num_features = len(features_cols)

    # 4. Решейп, если модель "сплющила" данные (80 * 67 = 5360)
    # Если в shap_matrix.shape[-1] число кратное num_features, восстанавливаем 3D
    if shap_matrix.ndim == 2 and shap_matrix.shape[1] > num_features:
        # Пробуем восстановить структуру (Samples, Seq_Len, Features)
        # Если 17152 / 67 не целое, берем остаток по num_features
        shap_matrix = shap_matrix[:, :seq_length * num_features]
        shap_matrix = shap_matrix.reshape(-1, seq_length, num_features)

    # 5. Агрегация: усредняем по батчу и временным шагам
    if shap_matrix.ndim == 3:
        # Усредняем по Samples (0) и TimeSteps (1)
        shap_abs = np.abs(shap_matrix).mean(axis=(0, 1))
    else:
        # Усредняем только по Samples
        shap_abs = np.abs(shap_matrix).mean(axis=0)

    # ❗ КРИТИЧЕСКОЕ ИСПРАВЛЕНИЕ: гарантируем 1D массив и нужную длину
    shap_abs = np.ravel(shap_abs) # Делает массив одномерным
    shap_abs = shap_abs[:num_features] # Отсекаем лишнее, если есть

    # Проверка перед созданием DF
    if len(shap_abs) == len(features_cols):
        df_importance = pd.DataFrame({
            'Feature': features_cols,
            'Importance': shap_abs
        }).sort_values(by='Importance', ascending=False)

        print(f"Успешно обработано признаков: {len(df_importance)}")
        print("Топ-5 важных признаков по SHAP:")
        print(df_importance.head(5).to_markdown(index=False))
    else:
        print(f"Ошибка: Не удалось сопоставить SHAP ({len(shap_abs)}) и признаки ({len(features_cols)})")

except Exception as e:
    print(f"Ошибка при расчете SHAP: {e}")

# --- 3. Загрузка бенчмарка и фильтрация (2026) ---
spy_data = yf.download('SPY', start=test_df_bench['Date'].min(), end=test_df_bench['Date'].max(),
                       interval='1d', auto_adjust=True, progress=False, multi_level_index=False)
spy_data.reset_index(inplace=True)
spy_data.rename(columns={'Close': 'SPY'}, inplace=True)


test_df_benchmark = pd.merge(test_df_bench, spy_data[['Date', 'SPY']], on='Date', how='inner')
test_df_benchmark['NVDA'] = test_df_benchmark['Close']
test_df_benchmark.set_index('Date', inplace=True)

# Очистка для ffn
test_df_benchmark = test_df_benchmark[(test_df_benchmark['NVDA'] > 0.01) &
                                     (test_df_benchmark['SPY'] > 0.01)].dropna()

# --- 4. Расчет доходностей ---
test_df_benchmark['NVDA_Returns'] = test_df_benchmark['NVDA'].pct_change()
test_df_benchmark['SPY_Returns'] = test_df_benchmark['SPY'].pct_change()
test_df_benchmark['Strategy_Returns'] = test_df_benchmark['NVDA_Returns'] * test_df_benchmark['Signal'].shift(1)
test_df_benchmark = test_df_benchmark.dropna()

# --- 5. Анализ производительности ---

prices_clean = test_df_benchmark[['NVDA', 'SPY']].copy()
prices_clean = prices_clean.replace(0, np.nan).dropna()

# 3. Расчет метрик
if not prices_clean.empty:
    perf = prices_clean.calc_stats()

    # Мы используем .stats.loc, чтобы гарантированно получить данные без ошибок атрибутов
    print("\n--- 📊 Итоговый отчет по активам (Январь 2026) ---")
    display_metrics = ['cagr', 'daily_sharpe', 'daily_vol', 'max_drawdown']
    summary = perf.stats.loc[display_metrics]
    summary.index = ['Годовая доходность', 'Коэф. Шарпа', 'Волатильность', 'Макс. просадка']
    print(summary.to_markdown())

    # --- 4. Финализация оптимального портфеля ---
    try:
        # Прямой расчет весов
        returns = prices_clean.to_returns().dropna()
        weights = returns.calc_mean_var_weights()

        # Расчет эквити (кривой капитала) портфеля
        port_returns = (returns * weights).sum(axis=1)
        port_equity = (1 + port_returns).cumprod()

        # Статистика портфеля
        port_stats = port_equity.calc_stats()

        print("\n--- ✨ Результат Оптимального Портфеля (2026) ---")
        print(f"Итоговый CAGR: {port_stats.cagr:.2%}")
        print(f"Итоговый Sharpe: {port_stats.daily_sharpe:.2f}")
        print(f"Максимальная просадка: {port_stats.max_drawdown:.2%}")

    except Exception as e:
        print(f"Ошибка финальной стадии: {e}")
else:
    print("Ошибка: Нет данных для анализа.")



--- 🧠 Вычисление SHAP Values ---
Успешно обработано признаков: 67
Топ-5 важных признаков по SHAP:
| Feature            |   Importance |
|:-------------------|-------------:|
| ADOSC_3_10         |  0.000385702 |
| MACD_6_13_5__Dbl1  |  0.000310972 |
| BBU_20_2.0_2.0     |  0.00024929  |
| pca_9              |  0.000242504 |
| MACDh_6_13_5__Dbl1 |  0.000227184 |

--- 📊 Итоговый отчет по активам (Январь 2026) ---
|                    |       NVDA |        SPY |
|:-------------------|-----------:|-----------:|
| Годовая доходность |  0.687672  |  0.229161  |
| Коэф. Шарпа        |  2.18147   |  2.53566   |
| Волатильность      |  0.281406  |  0.0910244 |
| Макс. просадка     | -0.0787389 | -0.0257846 |

--- ✨ Результат Оптимального Портфеля (2026) ---
Итоговый CAGR: 30.39%
Итоговый Sharpe: 2.10
Максимальная просадка: -4.26%
